# Build a game from the abstract classes

The companion `tutorial_create_full_prompt_new_game.ipynb` builds a new `FullPrompt` from the
abstract `PromptBlock`/`FullPrompt` base classes. This notebook does the same thing one level up:
it constructs a brand-new **game** - `SignChoiceGame` - directly from the abstract `Game` ABC
and its generic records (`GameSpec`, `AgentState`, `GameState`, `Observation`, `DecisionRequest`,
`Action`, `Transition`), the same shape `games/toy_coordination/game.py` and
`games/naming_convention/game.py` are both built from. Nothing here calls into an existing game -
every class is defined in this notebook, then proven to work by running it through the same
shared pieces every real game uses: the `loop_runtime` and the generic metrics.

`SignChoiceGame` is deliberately the smallest useful example: two agents are paired each round,
each privately chooses a signal (`X` or `Y`), and both score a point when their signals match.
That is exactly `toy_coordination`'s mechanic, built fresh here so every abstract class is visible
being assembled, not imported pre-made.

## Class map (what we are about to build, and against what abstraction)

| We define | Abstract base it implements | Also used, unmodified |
| --- | --- | --- |
| 4 concrete prompt blocks + `SignChoiceFullPrompt` | `PromptBlock[T]`, `FullPrompt` | `ResponseContract`, `UNBOUND` |
| `SignChoiceGame` | `Game` (ABC) | `GameSpec`, `AgentState`, `GameState`, `Observation`, `DecisionRequest`, `Action`, `Transition` (all generic, reused as-is) |
| — | — | `run_validated_decision` (`mas_cc.runtime`), `RoundView` + generic metrics (`mas_cc.metrics`) |

Nothing in the middle column gets modified for a new game; everything in it is already
game-agnostic. Only the left column is new.


In [15]:
from __future__ import annotations

import json
import os
from collections.abc import Mapping, Sequence
from dataclasses import dataclass, field, replace
from typing import Any

from mas_cc.config import GameConfig, LLMProviderConfig
from mas_cc.core import AgentId, InteractionId, MessageRole, Seed, ValidationIssue, ValidationResult
from mas_cc.games import Game
from mas_cc.games.protocols import Action, AgentState, DecisionRequest, GameSpec, GameState, Observation, Transition
from mas_cc.llm_providers import create_llm_provider
from mas_cc.metrics import ActionSharePerOption, DominantValueShare, FirstConsensusTime, RoundView
from mas_cc.prompts import UNBOUND, FullPrompt, PromptBlock, RegexTokenCounter, ResponseContract, Unbound
from mas_cc.runtime import run_validated_decision


## 1. The concrete prompt: four `PromptBlock` subclasses

Same pattern as the prompt tutorial - each block owns its own value, validation, and rendering.
Kept to four blocks (no per-agent private-state block) since the point of *this* notebook is the
whole game, not prompt depth; the prompt tutorial already covers building a richer, six-block
prompt in detail.


In [16]:
@dataclass(frozen=True, slots=True)
class DescriptionBlock(PromptBlock[str]):
    name: str = field(init=False, default="description")
    title: str = field(init=False, default="Description")
    role: MessageRole = field(init=False, default=MessageRole.SYSTEM)
    value: str | Unbound = "Choose the same signal as your partner to score a point."
    required: bool = field(init=False, default=True)
    binding: str = field(init=False, default="fixed")
    def value_issues(self, value):
        return () if isinstance(value, str) and value.strip() else (
            ValidationIssue("prompt.blocks.description.value", "must be non-empty text", value),
        )
    def render(self):
        return str(self.value)


@dataclass(frozen=True, slots=True)
class RulesBlock(PromptBlock[tuple]):
    name: str = field(init=False, default="rules")
    title: str = field(init=False, default="Rules")
    role: MessageRole = field(init=False, default=MessageRole.SYSTEM)
    value: tuple | Unbound = (
        "Choose exactly one signal: X or Y.",
        "You do not see your partner's current choice.",
        "You score 1 point when your signals match, 0 otherwise.",
    )
    required: bool = field(init=False, default=True)
    binding: str = field(init=False, default="fixed")
    def value_issues(self, value):
        ok = isinstance(value, Sequence) and not isinstance(value, (str, bytes)) and all(isinstance(x, str) and x for x in value)
        return () if ok else (ValidationIssue("prompt.blocks.rules.value", "must contain non-empty strings", value),)
    def render(self):
        return "\n".join(f"{i}. {r}" for i, r in enumerate(self.value, 1))


@dataclass(frozen=True, slots=True)
class AvailableSignalsBlock(PromptBlock[tuple]):
    name: str = field(init=False, default="available_signals")
    title: str = field(init=False, default="Available signals")
    role: MessageRole = field(init=False, default=MessageRole.USER)
    value: tuple | Unbound = UNBOUND
    required: bool = field(init=False, default=True)
    binding: str = field(init=False, default="dynamic")
    def value_issues(self, value):
        ok = isinstance(value, Sequence) and not isinstance(value, (str, bytes)) and len(value) >= 2
        return () if ok else (ValidationIssue("prompt.blocks.available_signals.value", "must contain at least two signals", value),)
    def render(self):
        return "Available signals: " + ", ".join(self.value)


@dataclass(frozen=True, slots=True)
class VisibleMemoryBlock(PromptBlock[tuple]):
    name: str = field(init=False, default="visible_memory")
    title: str = field(init=False, default="Recent memory")
    role: MessageRole = field(init=False, default=MessageRole.USER)
    value: tuple | Unbound = UNBOUND
    required: bool = field(init=False, default=True)
    binding: str = field(init=False, default="dynamic")
    def value_issues(self, value):
        ok = isinstance(value, Sequence) and not isinstance(value, (str, bytes)) and all(isinstance(x, Mapping) for x in value)
        return () if ok else (ValidationIssue("prompt.blocks.visible_memory.value", "must be a sequence of mappings", value),)
    def render(self):
        if not self.value:
            return "No previous interactions."
        return "\n".join(f"- {json.dumps(dict(e), sort_keys=True)}" for e in self.value)


## 2. `FullPrompt`: authoritative block order and the response contract

`FullPrompt` owns the ordered tuple of blocks and the `ResponseContract` that governs what a valid
answer looks like. `"choice_only"` here means the raw response text *is* the choice (`X` or `Y`),
unlike naming-convention's richer `paper_choice_reason` JSON contract.


In [17]:
class SignChoiceFullPrompt(FullPrompt):
    def concrete_prompt_type(self) -> str:
        return "sign_choice_decision"


def sign_choice_prompt() -> SignChoiceFullPrompt:
    return SignChoiceFullPrompt(
        family="sign_choice_decision",
        version=1,
        blocks=(DescriptionBlock(), RulesBlock(), AvailableSignalsBlock(), VisibleMemoryBlock()),
        response_contract=ResponseContract("choice_only", ("X", "Y"), {"decision_instruction": "Choose your signal now."}),
        message_mode="per_block",
    )


def bind_sign_prompt(*, memory: tuple[Mapping[str, Any], ...]) -> SignChoiceFullPrompt:
    """One immutable, independently-bound prompt per agent per decision."""
    return sign_choice_prompt().bind(available_signals=("X", "Y"), visible_memory=memory)


# Prove immutable binding + non-leakage before building anything else on top of it.
compiled_probe = bind_sign_prompt(memory=()).compile(RegexTokenCounter())
print(json.dumps({"blocks": [b["name"] for b in compiled_probe.blocks_as_dicts()], "total_tokens": compiled_probe.total_tokens}, indent=2))


{
  "blocks": [
    "description",
    "rules",
    "available_signals",
    "visible_memory"
  ],
  "total_tokens": 60
}


## 3. `GameSpec` and `initialize` — `AgentState`/`GameState` are already generic

`Game` is an `abc.ABC`, not a structural protocol - Python computes which abstract methods are
still missing once, when a class body finishes executing, and refuses to construct any subclass
that's still missing one. That means we can't build `SignChoiceGame` by patching methods onto an
already-created class one at a time (the usual "add one piece per cell" notebook trick) - by the
time we did that, the class would already be frozen as incomplete. So instead: each piece below is
a free function, explained in its own cell exactly as before, and the actual `class
SignChoiceGame(Game):` statement that assembles all of them happens once, in section 8, right
after the last piece exists.

`initialize` is the first proof that agent identity + memory need no new class at all: `AgentState`
from `games/protocols.py` already is that container.


In [18]:
sign_choice_spec = GameSpec(
    game_type="sign_choice",
    version=1,
    description="Pairwise agents choose signal X or Y and score a point when they match.",
    game_family="choice",
    minimum_population=2,
    supported_topologies=("complete",),
)


def initialize(self, config: GameConfig, seed: int) -> GameState:
    agents = tuple(
        AgentState(AgentId(f"agent-{index:03d}"), attributes={"available_signals": ["X", "Y"]})
        for index in range(config.population_size)
    )
    return GameState(game_type=self.spec.game_type, turn=0, agents=agents, data={"horizon": config.horizon})


## 4. `select_participants` and `construct_observations` — the information boundary

`select_participants` picks who acts this round; `construct_observations` decides what each of
them may see. This is where a game enforces its scientific information boundary - notice the
`Observation` below carries only `interaction_number`, nothing about the partner's identity or
choice.


In [19]:
def select_participants(self, state, config, rng):
    return tuple(rng.sample([agent.agent_id for agent in state.agents], k=2))


def construct_observations(self, state, participants, config):
    interaction_id = InteractionId(f"interaction-{state.turn + 1:04d}")
    return tuple(
        Observation(
            agent_id=agent_id, interaction_id=interaction_id, participants=participants,
            visible_state={"interaction_number": state.turn + 1},
        )
        for agent_id in participants
    )



## 5. `build_decision_requests` — where the concrete prompt gets bound in

This is the seam between the game and the prompt built in steps 1-2: each agent's own visible
memory gets bound into a fresh, independent `SignChoiceFullPrompt`, then wrapped in a
`DecisionRequest` alongside its `Observation`.


In [21]:
def _bound_prompt(self, state, observation):
    agent = state.agent(observation.agent_id)
    return bind_sign_prompt(memory=agent.memory)


def build_decision_requests(self, state, observations, config):
    return tuple(
        DecisionRequest(
            agent_id=observation.agent_id, interaction_id=observation.interaction_id,
            stage="simultaneous_choice", observation=observation,
            prompt=self._bound_prompt(state, observation), provider_required=True, retry_bound=1,
        )
        for observation in observations
    )



## 6. `parse_action` and `validate_action` — what a legal answer means, for this game

This is the only game-specific validity logic the shared `loop_runtime` ever calls into (see the
`loop_runtime`/metrics session notes on why this can't live in the provider or be duplicated per
game). Here it's almost trivially simple: the raw text *is* the signal.


In [22]:
def parse_action(self, request, response):
    return Action(agent_id=request.agent_id, value=response.strip(), stage=request.stage)


def validate_action(self, state, request, action, config):
    issues = []
    if action.value not in ("X", "Y"):
        issues.append(ValidationIssue("action.value", "must be exactly X or Y", action.value))
    return ValidationResult(tuple(issues))



## 7. `apply_transition` and `detect_termination` — pure, immutable state update

No provider, no randomness beyond what already happened. A new `GameState` comes back; only the
two participants' `memory`/`score` change.


In [23]:
def apply_transition(self, state, participants, actions, config):
    matched = actions[0].value == actions[1].value
    payoff = 1.0 if matched else 0.0
    action_by_agent = {action.agent_id: action.value for action in actions}
    updated_agents = []
    for agent in state.agents:
        if agent.agent_id not in participants:
            updated_agents.append(agent)
            continue
        other = next(p for p in participants if p != agent.agent_id)
        memory = (
            *agent.memory,
            {"own_signal": action_by_agent[agent.agent_id], "partner_signal": action_by_agent[other], "payoff": payoff},
        )
        updated_agents.append(replace(agent, score=agent.score + payoff, memory=memory))
    next_turn = state.turn + 1
    terminated = next_turn >= config.horizon
    next_state = GameState(
        game_type=state.game_type, turn=next_turn, agents=tuple(updated_agents),
        terminated=terminated, data=dict(state.data),
    )
    return Transition(
        interaction_id=InteractionId(f"interaction-{next_turn:04d}"), actions=actions,
        payoffs={str(agent_id): payoff for agent_id in participants}, next_state=next_state,
        matched=matched, termination_reason="finite_horizon_reached" if terminated else None,
    )


def detect_termination(self, state, config):
    return "finite_horizon_reached" if state.turn >= config.horizon else None



## 8. `call_plan`, then assembling `SignChoiceGame(Game)` in one place

`call_plan` is what Phase 4/9 pricing and preflight use to estimate token/cost demand *before* any
provider call - it never runs during actual play. This minimal version reuses one representative
prompt scenario for all three token bounds (lower/representative/maximum); a production game
should build a genuine maximum-memory scenario the way `games/toy_coordination/game.py:226` does.

Every piece now exists as a free name (`sign_choice_spec`, `initialize`, ..., `call_plan`), so this
is where we write the one `class SignChoiceGame(Game):` statement - referencing all of them as
class-body attributes - that Python's ABC machinery actually checks. `isinstance(game, Game)`
below is a real, enforced claim: if anything were still missing, the line right before it would
already have raised `TypeError` instead.


In [24]:
from mas_cc.planning import DecisionStagePlan, GameCallPlan, InteractionCount, PromptScenario


def call_plan(self, config):
    representative = bind_sign_prompt(memory=())
    return GameCallPlan(
        game_type=self.spec.game_type,
        game_version=self.spec.version,
        interactions=InteractionCount(
            fixed=config.horizon, lower=config.horizon, expected=config.horizon, maximum=config.horizon,
        ),
        decision_stages=(
            DecisionStagePlan(
                name="simultaneous_choice",
                requests_per_interaction=2,
                retry_bound=1,
                lower_prompt=PromptScenario("no_memory", representative, ("No interaction memory has accumulated.",)),
                representative_prompt=PromptScenario("no_memory", representative, ("No interaction memory has accumulated.",)),
                maximum_prompt=PromptScenario("no_memory", representative, ("This minimal example reuses the empty-memory scenario as its maximum; see toy_coordination for a real one.",)),
                prompt_scenarios=(PromptScenario("no_memory", representative),),
                assumptions=("Both selected agents require one independent decision.",),
            ),
        ),
        stopping_condition_assumptions=(f"The run stops after exactly {config.horizon} interactions.",),
        metadata={"population_size": config.population_size, "topology": config.topology, "provider_prices_included": False},
    )


class SignChoiceGame(Game):
    """Every piece defined above, assembled once - this is the statement Python's
    ABC machinery actually inspects."""

    spec = sign_choice_spec
    initialize = initialize
    select_participants = select_participants
    construct_observations = construct_observations
    _bound_prompt = _bound_prompt
    build_decision_requests = build_decision_requests
    parse_action = parse_action
    validate_action = validate_action
    apply_transition = apply_transition
    detect_termination = detect_termination
    call_plan = call_plan


game = SignChoiceGame()
print("satisfies the Game ABC:", isinstance(game, Game))


satisfies the Game ABC: True


## 9. Exercise it: one full decision, through the shared `loop_runtime`

Everything above never touched a provider. Now we run one real decision through
`run_validated_decision` - the same shared ask/validate/retry loop `naming_convention` and
`toy_coordination` both use - against the real University proxy. Requires
`POTSDAM_API_KEY`/`BASE_POTSDAM_LLM_URL` in your environment (or repo-root `.env`).


In [25]:
config = GameConfig(type="sign_choice", population_size=4, horizon=6)
state = game.initialize(config, seed=0)
root_seed = Seed(0)
counter = RegexTokenCounter()

print(json.dumps({
    "agents": [str(agent.agent_id) for agent in state.agents],
    "agent_000_memory": list(state.agents[0].memory),
    "agent_000_available_signals": list(state.agents[0].attributes.get("available_signals", ())),
}, indent=2))

pair_rng = root_seed.derive("pair-sampling:1").create_random()
pair = game.select_participants(state, config, pair_rng)
observations = game.construct_observations(state, pair, config)
requests = game.build_decision_requests(state, observations, config)
compiled_prompts = [request.prompt.compile(counter) for request in requests]

print("pair:", [str(agent_id) for agent_id in pair])
for request, compiled in zip(requests, compiled_prompts, strict=True):
    print(json.dumps({
        "agent_id": str(request.agent_id),
        "blocks": [block["name"] for block in compiled.blocks_as_dicts()],
        "definition_hash": compiled.definition_hash[:12],
        "instance_hash": compiled.instance_hash[:12],
    }, indent=2))


{
  "agents": [
    "agent-000",
    "agent-001",
    "agent-002",
    "agent-003"
  ],
  "agent_000_memory": [],
  "agent_000_available_signals": [
    "X",
    "Y"
  ]
}
pair: ['agent-003', 'agent-002']
{
  "agent_id": "agent-003",
  "blocks": [
    "description",
    "rules",
    "available_signals",
    "visible_memory"
  ],
  "definition_hash": "09d2fd683247",
  "instance_hash": "c17a3a8a2ec5"
}
{
  "agent_id": "agent-002",
  "blocks": [
    "description",
    "rules",
    "available_signals",
    "visible_memory"
  ],
  "definition_hash": "09d2fd683247",
  "instance_hash": "c17a3a8a2ec5"
}


In [26]:
assert bool(os.getenv("POTSDAM_API_KEY")) and bool(os.getenv("BASE_POTSDAM_LLM_URL")), (
    "Set POTSDAM_API_KEY / BASE_POTSDAM_LLM_URL (e.g. in a repo-root .env) before running this cell."
)
university_provider = create_llm_provider(
    LLMProviderConfig(
        type="university", model=os.getenv("POTSDAM_MODEL", "gwdg/qwen3-30b-a3b-instruct-2507"),
        credentials_env="POTSDAM_API_KEY", base_url_env="BASE_POTSDAM_LLM_URL", max_output_tokens=16,
    ),
    environment=os.environ,
)
decisions = []
for request, compiled in zip(requests, compiled_prompts, strict=True):
    decision = await run_validated_decision(
        game=game, state=state, request=request, game_config=config,
        provider=university_provider, prompt=compiled,
        temperature=0.0, max_output_tokens=16,
        seed_for_attempt=lambda attempt, r=request: int(root_seed.derive(f"{r.interaction_id}:{r.agent_id}:{attempt}")),
        metadata_for_attempt=lambda attempt, r=request: {"agent_id": str(r.agent_id), "attempt": attempt + 1},
    )
    decisions.append(decision)
    print(json.dumps({
        "agent_id": str(request.agent_id), "action": decision.action.value,
        "attempts": len(decision.attempts), "raw_response": decision.attempts[-1].response.content,
    }, indent=2))

transition = game.apply_transition(state, pair, tuple(decision.action for decision in decisions), config)
state = transition.next_state
print(json.dumps({"matched": transition.matched, "payoffs": dict(transition.payoffs)}, indent=2))
university_provider.close()


{
  "agent_id": "agent-003",
  "action": "X",
  "attempts": 1,
  "raw_response": "X"
}
{
  "agent_id": "agent-002",
  "action": "X",
  "attempts": 1,
  "raw_response": "X"
}
{
  "matched": true,
  "payoffs": {
    "agent-003": 1.0,
    "agent-002": 1.0
  }
}


## 10. Metrics: the same `RoundView` every game adapts into

A `to_round_view`-style adapter for a brand-new game is this small - read each agent's last played
signal from its own memory, and hand over the game's option set. Everything else
(`ActionSharePerOption`, `DominantValueShare`, `FirstConsensusTime`, ...) comes straight off the
shelf from `mas_cc.metrics`, unmodified.

Passing `options` matters: it is what makes `ActionSharePerOption` emit a row for **every** signal
each round, including one nobody has chosen yet, so a share of 0 is a real measurement rather than
a missing one. Because `SignChoiceGame` declares `game_family="choice"`, that metric is legal here -
`games/registry.py::game_metrics` would reject it on a game whose agents report numbers, not
options.


In [27]:
def to_round_view(state) -> RoundView:
    return RoundView(
        agent_values={
            agent.agent_id: (agent.memory[-1]["own_signal"] if agent.memory else None)
            for agent in state.agents
        },
        options=("X", "Y"),
    )


view = to_round_view(state)
print("round view:", {str(agent_id): value for agent_id, value in view.agent_values.items()})
print("population_action_share_per_option:", ActionSharePerOption().compute_round(view))
print("dominant_value_share:", DominantValueShare().compute_round(view))


round view: {'agent-000': None, 'agent-001': None, 'agent-002': 'X', 'agent-003': 'X'}
population_action_share_per_option: {'X': 1.0, 'Y': 0.0}
dominant_value_share: {None: 1.0}


## 11. A full trajectory, then metrics over it

Run the rest of the horizon the same way, one round at a time, then fold `RoundView` over every
resulting state and compute a final metric (`FirstConsensusTime`) over the whole trajectory.


In [28]:
university_provider = create_llm_provider(
    LLMProviderConfig(
        type="university", model=os.getenv("POTSDAM_MODEL", "gwdg/qwen3-30b-a3b-instruct-2507"),
        credentials_env="POTSDAM_API_KEY", base_url_env="BASE_POTSDAM_LLM_URL", max_output_tokens=16,
    ),
    environment=os.environ,
)
views = [view]
try:
    for round_index in range(state.turn + 1, config.horizon + 1):
        rng = root_seed.derive(f"pair-sampling:{round_index}").create_random()
        pair = game.select_participants(state, config, rng)
        observations = game.construct_observations(state, pair, config)
        requests = game.build_decision_requests(state, observations, config)
        round_decisions = []
        for request in requests:
            compiled = request.prompt.compile(counter)
            decision = await run_validated_decision(
                game=game, state=state, request=request, game_config=config,
                provider=university_provider, prompt=compiled,
                temperature=0.0, max_output_tokens=16,
                seed_for_attempt=lambda attempt, r=request: int(root_seed.derive(f"{r.interaction_id}:{r.agent_id}:{attempt}")),
                metadata_for_attempt=lambda attempt, r=request: {"agent_id": str(r.agent_id), "attempt": attempt + 1},
            )
            round_decisions.append(decision)
        transition = game.apply_transition(state, pair, tuple(d.action for d in round_decisions), config)
        state = transition.next_state
        views.append(to_round_view(state))
finally:
    university_provider.close()

print("final scores:", {str(agent.agent_id): agent.score for agent in state.agents})
print("first_consensus_time_by_action_share:",
      FirstConsensusTime(threshold=0.75).compute_final(tuple(views)))


final scores: {'agent-000': 4.0, 'agent-001': 3.0, 'agent-002': 4.0, 'agent-003': 1.0}
first_consensus_time_by_action_share: 1


## 12. Promotion-to-production checklist

1. Move the four blocks, `SignChoiceFullPrompt`, and `SignChoiceGame` to `games/sign_choice/{prompts,game}.py`
   (see `games/toy_coordination/` for the same shape at production scale).
2. Replace the minimal `call_plan` (section 8) with a real maximum-memory scenario, following
   `games/toy_coordination/game.py:226`.
3. Register the game in `create_default_game_registry()` and add a `configs/components/games/sign_choice.yaml`.
4. Write a `to_round_view` module-level adapter (as in section 10) plus a `METRICS` list, following
   `games/naming_convention/metrics.py`.
5. Add contract tests: satisfies `Game`, deterministic transitions, immutable binding, no
   cross-agent memory leakage, golden prompt/message fixtures.
6. Wire a `runtime.py` if this game needs anything `games/runner.py`'s generic loop doesn't already
   provide (concurrency shape, richer audit records) - most games shouldn't need one.
